# Week 58, Day 6: Forge: Week 58 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w058/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
p = 30 / 40
print("1.", round(1 - p ** 2 - (1 - p) ** 2, 4))
print("2.", round((1 - 1 / 1000) ** 1000, 4), "close to 1/e =", round(np.exp(-1), 4))
print("3.", round(0.5 * np.log(0.7 / 0.3), 4))

### Exercise: Part A: linear, forest and boosted trees

Complete fit_tree (a regression tree of limited depth fitted to residuals) and the boosting loop: update the fitting-set and validation-set predictions, track validation MSE, and refit on the full training set for best_round rounds. Then report the test R squared of all three models.

In [ ]:
import numpy as np
r = np.random.default_rng(586)

def session(n, g):
    X = g.uniform(-1, 1, (n, 6))
    rate = (8 + 3 * np.sin(2 * X[:, 0]) + 2 * X[:, 1] ** 2 + 4 * (X[:, 2] > 0.3)
            + 4 * X[:, 0] * (X[:, 3] > 0))
    return X, rate + 1.0 * g.normal(size=n)

def split(X, y, feats, min_leaf=3):
    n, best = len(y), (None, 0.0, 0.0)
    for j in feats:
        o = np.argsort(X[:, j]); xs, ys = X[o, j], y[o]
        cs = np.cumsum(ys)[:-1]; nl = np.arange(1, n)
        gain = cs ** 2 / nl + (ys.sum() - cs) ** 2 / (n - nl) - ys.sum() ** 2 / n
        ok = (xs[1:] > xs[:-1]) & (nl >= min_leaf) & (n - nl >= min_leaf)
        gain = np.where(ok, gain, -np.inf); k = int(np.argmax(gain))
        if gain[k] > best[2]: best = (j, (xs[k] + xs[k + 1]) / 2, gain[k])
    return best[:2]

def fit_tree(X, y, depth, g=None, mtry=None):
    if depth == 0 or len(y) < 6: return ("leaf", y.mean())
    feats = range(X.shape[1]) if mtry is None else g.choice(X.shape[1], mtry, replace=False)
    j, t = split(X, y, feats)
    if j is None: return ("leaf", y.mean())
    m = X[:, j] <= t
    return ("node", j, t, ..., ...)

def predict(tree, X):
    if tree[0] == "leaf": return np.full(len(X), tree[1])
    _, j, t, L, R = tree; m = X[:, j] <= t; out = np.empty(len(X))
    out[m], out[~m] = predict(L, X[m]), predict(R, X[~m]); return out

r2 = lambda y_, p: float(1 - np.mean((y_ - p) ** 2) / np.var(y_))
X, y = session(600, r); X_te, y_te = session(3000, r)
scores = {}

# linear model by least squares
A = np.column_stack([np.ones(600), X]); w = np.linalg.lstsq(A, y, rcond=None)[0]
scores["linear"] = r2(y_te, np.column_stack([np.ones(3000), X_te]) @ w)

# random forest: 40 deep trees, 3 features per split
gf = np.random.default_rng(1); forest = []
for b in range(40):
    idx = gf.integers(0, 600, 600)
    forest.append(fit_tree(X[idx], y[idx], 20, gf, 3))
scores["forest"] = r2(y_te, np.mean([predict(t, X_te) for t in forest], axis=0))

# gradient boosting with depth-2 trees: early stopping on a validation split
nu, fi, vi = 0.1, np.arange(450), np.arange(450, 600)
F, Fv, val_mse = np.full(450, y[fi].mean()), np.full(150, y[fi].mean()), []
for m in range(600):
    h = fit_tree(X[fi], y[fi] - F, 2)
    F += nu * predict(h, X[fi])
    Fv += ...
    val_mse.append(np.mean((y[vi] - Fv) ** 2))
best_round = int(np.argmin(val_mse)) + 1

F, F_te = np.full(600, y.mean()), np.full(3000, y.mean())
for m in range(best_round):
    h = fit_tree(X, y - F, 2)
    F += nu * predict(h, X)
    F_te += nu * predict(h, X_te)
scores["boosted"] = r2(y_te, F_te)
for k, v in scores.items():
    print(f"{k:8s} test R squared {v:.3f}")
print(f"boosting stopped at round {best_round}; noise ceiling about {1 - 1 / np.var(y_te):.3f}")

In [ ]:
import numpy as np
def test_models():
    "test R squared is recorded for all three models"
    assert set(scores) == {"linear", "forest", "boosted"}
def test_trees_win():
    "both tree ensembles beat the linear model by at least 0.1"
    assert scores["forest"] > scores["linear"] + 0.1 and scores["boosted"] > scores["linear"] + 0.1
def test_stop():
    "the number of rounds was chosen by validation, between 20 and 600"
    assert 20 <= best_round <= 600

test_models()
test_trees_win()
test_stop()
print("All checks passed")

### Exercise: Part B: the training-error bound

Run AdaBoost for 100 rounds on 300 noise-free ring examples. After each round, record the training error of the ensemble and the running product of 2 sqrt(eps (1 - eps)).

In [ ]:
def ring(n, g):
    X = g.uniform(-1, 1, (n, 2))
    return X, np.where(X[:, 0] ** 2 + X[:, 1] ** 2 < 0.45, 1.0, -1.0)

def weighted_stump(X, y, w):
    best = (0, 0.0, 1, np.inf)
    for j in range(X.shape[1]):
        o = np.argsort(X[:, j]); xs, ys, ws = X[o, j], y[o], w[o]
        pl = np.r_[0, np.cumsum(ws * (ys > 0))]; nl_ = np.r_[0, np.cumsum(ws * (ys < 0))]
        for s, err in [(1, pl + (nl_[-1] - nl_)), (-1, (pl[-1] - pl) + nl_)]:
            k = int(np.argmin(err))
            t = xs[0] - 1 if k == 0 else (xs[-1] + 1 if k == len(xs) else (xs[k - 1] + xs[k]) / 2)
            if err[k] < best[3]: best = (j, t, s, err[k])
    return best

Xb, yb = ring(300, np.random.default_rng(7))
w, Fb, train_err, bound, prod = np.ones(300) / 300, np.zeros(300), [], [], 1.0
for t_ in range(100):
    j, t, s, eps = weighted_stump(Xb, yb, w)
    alpha = 0.5 * np.log((1 - eps) / eps)
    h = np.where(Xb[:, j] > t, s, -s)
    w = w * np.exp(-alpha * yb * h); w /= w.sum()
    Fb += alpha * h
    prod *= ...
    train_err.append(float(np.mean(np.sign(Fb) != yb))); bound.append(prod)
for k in (0, 9, 29, 99):
    print(f"round {k + 1:3d}: training error {train_err[k]:.3f}, bound {bound[k]:.3f}")

In [ ]:
import numpy as np
def test_bound():
    "at every round the training error is at most the product bound"
    assert len(train_err) == 100 and all(e <= b + 1e-9 for e, b in zip(train_err, bound))
def test_falls():
    "the bound falls below 0.3 within 100 rounds"
    assert bound[-1] < 0.3

test_bound()
test_falls()
print("All checks passed")